In [ ]:
# Quicktest env: ставим только нужное (минимум изменений окружения)
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

REQS = {
    "transformers": "4.51.3",
    "tokenizers": "0.21.1",
    "datasets": "3.6.0",
    "accelerate": "1.1.1",
    "huggingface-hub": "0.35.3",
    "scikit-learn": "1.5.2",
    "matplotlib": "3.8.4",
}

def get_ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

to_install = []
for pkg, min_ver in REQS.items():
    cur = get_ver(pkg)
    if cur is None or Version(cur) < Version(min_ver):
        to_install.append(f"{pkg}>={min_ver}")

if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *to_install])
    print("Installed:", ", ".join(to_install))
else:
    print("Environment OK")

In [ ]:
import os, re, html, random, math, warnings
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback
)
warnings.filterwarnings('ignore', category=FutureWarning)
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# QUICK CONFIG (быстрый и максимально стабильный smoke-test)
POS_PATH = 'crypto_twitter_dataset2.csv'
NEG_PATH = 'non_crypto_tweets.csv'
OUT_ROOT = './models/crypto-detector-deberta-v3-large-quicktest'
MODEL_NAME = 'microsoft/deberta-v3-large'

SEED = 42
MAX_LEN = 96
EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 8e-5   # для head-only обучения
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 0.3
MAX_SAMPLES_PER_CLASS = 700
MAX_STEPS = 80

# Стабильный режим для проверки метрик
FP16, BF16 = False, False

os.makedirs(OUT_ROOT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Model:', MODEL_NAME)
print('MAX_SAMPLES_PER_CLASS:', MAX_SAMPLES_PER_CLASS)
print('MAX_STEPS:', MAX_STEPS)
print('fp16/bf16:', FP16, BF16)

In [ ]:
SEARCH_ROOTS = [Path('.'), Path('/kaggle/input'), Path('/kaggle/working'), Path('/content')]

def find_file_by_name(filename: str):
    p = Path(filename)
    if p.exists():
        return p

    # точный поиск по имени
    for root in SEARCH_ROOTS:
        if root.exists():
            hits = list(root.rglob(p.name))
            if hits:
                return hits[0]

    # мягкий поиск по похожему stem
    key = p.stem.lower().replace('-', '').replace('_', '')
    for root in SEARCH_ROOTS:
        if root.exists():
            for f in root.rglob('*.csv'):
                stem = f.stem.lower().replace('-', '').replace('_', '')
                if key in stem or stem in key:
                    return f
    return None

def clean_text(x: str) -> str:
    if not isinstance(x, str):
        return ''
    x = html.unescape(x)
    x = re.sub(r'https?://\S+', ' ', x)
    x = re.sub(r'@\w+', ' ', x)
    x = re.sub(r'#\w+', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

def read_any_csv(path: str, label: int) -> pd.DataFrame:
    found = find_file_by_name(path)
    if found is None:
        raise FileNotFoundError(f'Не найден файл: {path}')

    # сначала ';' (наши датасеты в основном такие), затем auto
    for sep in [';', None, ',']:
        try:
            df = pd.read_csv(found, sep=sep, engine='python', on_bad_lines='skip', encoding_errors='replace')
            col = next((c for c in ['tweet_text', 'full_text', 'text'] if c in df.columns), None)
            if col:
                out = pd.DataFrame({'text': df[col].astype(str).map(clean_text), 'label': int(label)})
                print(f'Loaded {found} with sep={sep!r}, rows={len(out)}, text_col={col}')
                return out
        except Exception:
            pass
    raise RuntimeError(f'Cannot read csv: {found}')

def normalize(df):
    # критично: удаляем мусорные/пустые/аномально длинные строки
    df = df.copy()
    df = df[df['text'].notna()]
    df['text'] = df['text'].astype(str).str.strip()
    df = df[(df['text'].str.len() >= 8) & (df['text'].str.len() <= 800)]
    # хотя бы одна буква/цифра
    df = df[df['text'].str.contains(r'[\w\d]', regex=True)]
    df['norm'] = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
    # quicktest: убираем явные дубли для стабильности
    df = df.drop_duplicates(subset=['norm']).reset_index(drop=True)
    return df

pos = normalize(read_any_csv(POS_PATH, 1))
neg = normalize(read_any_csv(NEG_PATH, 0))

overlap = set(pos['norm']).intersection(set(neg['norm']))
pos = pos[~pos['norm'].isin(overlap)].reset_index(drop=True)
neg = neg[~neg['norm'].isin(overlap)].reset_index(drop=True)

n = min(len(pos), len(neg), MAX_SAMPLES_PER_CLASS)
if n < 50:
    raise RuntimeError(f'Слишком мало валидных данных после нормализации: n={n}. Проверьте входные CSV.')
pos = pos.sample(n, random_state=SEED)
neg = neg.sample(n, random_state=SEED)
all_df = pd.concat([pos[['text','label']], neg[['text','label']]], ignore_index=True).sample(frac=1, random_state=SEED)

train_df, temp_df = train_test_split(all_df, test_size=0.2, random_state=SEED, stratify=all_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label'])

print('n/class:', n, 'overlap removed:', len(overlap))
print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('label dist train:', train_df['label'].value_counts().to_dict())
print('sample positive:', pos['text'].iloc[0][:180])
print('sample negative:', neg['text'].iloc[0][:180])

ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True)),
    'val': Dataset.from_pandas(val_df[['text','label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['text','label']].reset_index(drop=True)),
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN, padding=False)

ds_tok = ds.map(tok, batched=True)
for split in ['train','val','test']:
    drop_cols = [c for c in ds_tok[split].column_names if c not in {'input_ids','attention_mask','token_type_ids','label'}]
    if drop_cols:
        ds_tok[split] = ds_tok[split].remove_columns(drop_cols)
    # дополнительный фильтр на очень короткие токенизированные примеры
    ds_tok[split] = ds_tok[split].filter(lambda ex: len(ex['input_ids']) >= 3)

collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True, problem_type='single_label_classification', use_safetensors=False
)
model.config.id2label = {0:'non_crypto',1:'crypto'}
model.config.label2id = {'non_crypto':0,'crypto':1}
if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False
model = model.float()

# Quicktest: обучаем только head, чтобы избежать NaN и ускорить прогон
for p in model.parameters():
    p.requires_grad = False
for name, p in model.named_parameters():
    if ('classifier' in name) or ('pooler' in name):
        p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable}/{total} ({100*trainable/total:.4f}%)')

# Быстрая проверка на NaN до обучения (forward + backward на 1 батче)
probe = collator([ds_tok['train'][i] for i in range(min(8, len(ds_tok['train'])) )])
probe = {k: v.to(model.device) for k, v in probe.items()}
model.train()
probe_out = model(**probe)
print('Probe loss finite:', bool(torch.isfinite(probe_out.loss).item()))
probe_out.loss.backward()
bad_grad = False
for p in model.parameters():
    if p.requires_grad and p.grad is not None and (not torch.isfinite(p.grad).all()):
        bad_grad = True
        break
model.zero_grad(set_to_none=True)
if bad_grad:
    raise RuntimeError('Обнаружены NaN/Inf в градиентах на probe-батче. Вероятна ошибка в данных/окружении.')

def compute_metrics_from_logits(logits, labels):
    logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = (preds == labels).mean()
    return {'accuracy': float(acc), 'precision': float(p), 'recall': float(r), 'f1': float(f1)}

steps_per_epoch = math.ceil(len(ds_tok['train']) / (BATCH_SIZE * GRAD_ACCUM_STEPS))
warmup_steps = int(max(steps_per_epoch * EPOCHS, 1) * WARMUP_RATIO)

args = TrainingArguments(
    output_dir=OUT_ROOT,
    eval_strategy='no',
    save_strategy='no',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=FP16,
    bf16=BF16,
    logging_steps=10,
    logging_nan_inf_filter=True,
    report_to=[],
    lr_scheduler_type='cosine',
    remove_unused_columns=True,
    disable_tqdm=True,
    dataloader_num_workers=2,
    max_steps=MAX_STEPS,
    optim='adamw_torch',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok['train'],
    data_collator=collator,
 )

train_result = trainer.train()
print('Train finished. train_loss =', float(train_result.training_loss))

# Оценка без trainer.evaluate(), чтобы обойти notebook-callback баг
def manual_eval(dataset_split):
    model.eval()
    dl = torch.utils.data.DataLoader(dataset_split, batch_size=8, shuffle=False, collate_fn=collator)
    all_logits, all_labels = [], []
    losses = []
    with torch.no_grad():
        for b in dl:
            labels = b['labels']
            b = {k: v.to(model.device) for k, v in b.items()}
            out = model(**b)
            loss_val = float(out.loss.detach().cpu())
            if np.isfinite(loss_val):
                losses.append(loss_val)
            all_logits.append(out.logits.detach().cpu().numpy())
            all_labels.append(labels.numpy())
    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    m = compute_metrics_from_logits(logits, labels)
    m['loss'] = float(np.mean(losses)) if losses else float('nan')
    m['non_finite_logits'] = bool((~np.isfinite(logits)).any())
    return m

val_metrics = manual_eval(ds_tok['val'])
test_metrics = manual_eval(ds_tok['test'])
print('Validation:', val_metrics)
print('Test:', test_metrics)

if test_metrics.get('f1', 0.0) <= 0.0:
    print('WARNING: F1 is still zero. Full run is not recommended yet.')
else:
    print('OK: non-zero F1. Можно запускать полную версию.')